# Pipeline scRNA-seq: scanpy + SCENIC

**Versión 1.1.1**

### Un solo notebook, dos modos de datos — elige uno y ejecuta todo

Corre el pipeline completo (**clustering + 12 gráficas de análisis + redes SCENIC**) sobre uno de dos conjuntos de datos, según lo que elijas en **una sola celda** al inicio:

| Modo | Datos | Qué hace |
|---|---|---|
| **`"ejemplo"`** | Demostración | Médula ósea humana + SCENIC sobre dataset `tiny` |
| **`"zenodo"`** | Lupus real | Datos de Jang *et al.* + figuras del artículo + SCENIC sobre células B |

**Las 12 gráficas** (violines QC, scatter, HVG, PCA, grafo de vecinos, UMAP de QC, anotación, marcadores, DE, top-100 DE, heatmap, trayectoria) se generan **en ambos modos**.

**Cómo usarlo:** en la celda **1** pon `MODO = "ejemplo"` o `"zenodo"` → *Entorno de ejecución → Ejecutar todas*. Las celdas del otro modo se saltan solas.

> Modo `"zenodo"`: usa RAM alta si vas a procesar todas las células; con el downsample por defecto va en Colab gratis.

<details>
<summary><b>Historial de versiones</b> (clic para expandir)</summary>

| Versión | Cambios |
|---|---|
| **1.1.1** | Fix SCENIC (`np.object` roto en cisTarget), DE sin genes housekeeping, heatmap legible (gráfica 11) |
| 1.1.0 | Agregadas las 12 gráficas de análisis (QC, PCA, marcadores, DE, top-100, heatmap, trayectoria) en ambos modos |
| 1.0.0 | Notebook inicial: dos modos (`ejemplo`/`zenodo`), conversión RDS→h5ad con ahorro de RAM, SCENIC básico |

</details>


---
# 1 · Configuración

**Única celda que debes editar.** Elige el modo y ajusta los parámetros.


In [ ]:
# ============================================================
#   CELDA MAESTRA  —  elige que correr
# ============================================================
MODO = "ejemplo"           # "ejemplo"  o  "zenodo"

# ---- Solo para MODO="zenodo" -------------------------------
N_CELLS_MAX       = 60000   # celulas al convertir el RDS (0 = todas; requiere RAM alta)

# ---- SCENIC (Parte B) --------------------------------------
SCENIC_DOWNSAMPLE = True     # True = rapido; False = TODAS las celulas B (lento, horas)
SCENIC_N_CELLS    = 2000
SCENIC_N_GENES    = 1500

# ---- Trayectoria (grafica 12) ------------------------------
ROOT_CLUSTER      = None     # cluster raiz del pseudotiempo (str, p.ej. "0"); None = automatico

# ---- Clustering usado para DE / anotacion / trayectoria ----
CLUSTER_KEY       = "leiden_res_0.50"
# ============================================================
assert MODO in ("ejemplo","zenodo")
print("MODO:", MODO)


---
# 2 · Preparar el entorno
Instala las librerías (común a ambos modos). Si Colab pide reiniciar, hazlo y vuelve a *Ejecutar todas*.


### 2.1 · Memoria disponible

In [ ]:
import psutil, os
ram_gb = psutil.virtual_memory().total/1e9
print(f"RAM: {ram_gb:.1f} GB | CPUs: {os.cpu_count()}")
if MODO=="zenodo" and ram_gb<20 and N_CELLS_MAX==0:
    print("AVISO: pediste todas las celulas con poca RAM. Activa RAM alta o usa N_CELLS_MAX=60000.")


### 2.2 · Instalar scanpy

In [ ]:
!pip install -q scanpy leidenalg igraph scikit-misc fa2-modified 2>/dev/null
print("scanpy listo")


### 2.3 · Instalar pySCENIC

In [ ]:
!pip install -q pyscenic==0.12.1 2>/dev/null
try:
    import pyscenic, ctxcore, loompy
    print("pyscenic:", pyscenic.__version__)
except Exception as e:
    print("Si falla, reinicia el entorno y re-ejecuta.\n", e)


### 2.4 · (Opcional) Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR='/content/drive/MyDrive/scanpy_scenic_lupus'; os.makedirs(PROJECT_DIR, exist_ok=True)
except Exception:
    PROJECT_DIR='.'
print("Resultados en:", PROJECT_DIR)


---
# 3 · Carga de datos

Produce el objeto **`adata`** (crudo, con conteos) según el modo. El análisis y las 12 gráficas (sección 4) corren luego sobre `adata` en ambos modos.


## 3A · [MODO EJEMPLO] Cargar médula ósea
> Solo corre si `MODO=="ejemplo"`.


In [ ]:
if MODO=="ejemplo":
    import scanpy as sc, anndata as ad, numpy as np, pandas as pd, pooch
    sc.settings.verbosity=1; sc.settings.set_figure_params(dpi=70, facecolor='white')
    np.random.seed(0)
    EX = pooch.create(path=pooch.os_cache('scverse_tutorials'),
                      base_url='doi:10.6084/m9.figshare.22716739.v1/')
    EX.load_registry_from_doi()
    samples={'s1d1':'s1d1_filtered_feature_bc_matrix.h5','s1d3':'s1d3_filtered_feature_bc_matrix.h5'}
    adatas={}
    for sid,fn in samples.items():
        a=sc.read_10x_h5(EX.fetch(fn)); a.var_names_make_unique(); adatas[sid]=a
    adata=ad.concat(adatas, label='sample'); adata.obs_names_make_unique()
    col_cond=col_time=col_ctype=None   # no aplican en ejemplo
    print("adata:", adata.shape)


## 3B · [MODO ZENODO] Descargar y convertir los datos de lupus
> Solo corre si `MODO=="zenodo"`. Conversión R→Python en proceso separado (ahorra RAM).


### 3B.1 · Descargar RDS de Zenodo (1.6 GB)

In [ ]:
if MODO=="zenodo":
    import os
    RDS='RTX_zenodo.RDS'
    if not os.path.exists(RDS):
        print("Descargando (1.6 GB)...")
        !wget -q --show-progress -O {RDS} "https://zenodo.org/records/17868028/files/RTX_zenodo.RDS?download=1"
    print(f"{os.path.getsize(RDS)/1e9:.2f} GB")


### 3B.2 · Instalar SeuratObject en R

In [ ]:
if MODO=="zenodo":
    !Rscript -e 'if(!requireNamespace("SeuratObject",quietly=TRUE))install.packages("SeuratObject",repos="https://cloud.r-project.org"); if(!requireNamespace("Matrix",quietly=TRUE))install.packages("Matrix",repos="https://cloud.r-project.org"); cat("listo\n")'


### 3B.3 · Escribir el script de conversión

In [ ]:
%%writefile convert_rds.R
suppressMessages({library(SeuratObject); library(Matrix)})
n_max <- as.integer(Sys.getenv("N_CELLS_MAX","60000"))
obj <- readRDS("RTX_zenodo.RDS")
cat("=== ESTRUCTURA ===\n"); print(obj)
cat("\n=== COLUMNAS METADATA ===\n"); print(colnames(obj@meta.data))
meta <- obj@meta.data
cat("\n=== GRUPOS ===\n")
for (col in colnames(meta)) { v<-meta[[col]]
  if (is.factor(v)||is.character(v)||(is.numeric(v)&&length(unique(v))<30))
    if (length(unique(v))<=30){cat("\n[",col,"]\n",sep="");print(table(v))} }
ncells<-ncol(obj)
if (n_max>0 && n_max<ncells){ set.seed(0)
  cc<-grep("celltype|cell_type|cell.type|annotation|ident",colnames(meta),ignore.case=TRUE,value=TRUE)
  if(length(cc)>0){grp<-as.character(meta[[cc[1]]]);fr<-n_max/ncells
    idx<-sort(unlist(lapply(split(seq_len(ncells),grp),function(ix)sample(ix,max(1,round(length(ix)*fr))))))
  } else idx<-sort(sample(ncells,n_max))
  cat("\n>>> DOWNSAMPLE:",ncells,"->",length(idx),"\n")
} else { idx<-seq_len(ncells); cat("\n>>> TODAS:",ncells,"\n") }
ar<-if("RNA"%in%Assays(obj))"RNA" else DefaultAssay(obj)
counts<-tryCatch(GetAssayData(obj,assay=ar,slot="counts"),
                 error=function(e)GetAssayData(obj,assay=ar,layer="counts"))[,idx]
Matrix::writeMM(counts,"counts.mtx")
write.csv(data.frame(gene=rownames(counts)),"genes.csv",row.names=FALSE)
write.csv(data.frame(barcode=colnames(counts)),"barcodes.csv",row.names=FALSE)
write.csv(meta[idx,,drop=FALSE],"metadata.csv")
reds<-Reductions(obj); un<-reds[grepl("umap",tolower(reds))][1]; if(is.na(un))un<-reds[1]
write.csv(Embeddings(obj,un)[idx,,drop=FALSE],"umap.csv")
rm(obj,counts,meta); gc(); cat("\nEmbedding:",un,"\nListo.\n")


### 3B.4 · Ejecutar la conversión (lee el output: ahí están los grupos)

In [ ]:
if MODO=="zenodo":
    import os
    os.environ['N_CELLS_MAX']=str(N_CELLS_MAX)
    !N_CELLS_MAX=$N_CELLS_MAX Rscript convert_rds.R


### 3B.5 · Reconstruir `adata` (preservando el UMAP de los autores)

In [ ]:
if MODO=="zenodo":
    import scipy.io, gc, scanpy as sc, anndata as ad, pandas as pd, numpy as np
    sc.settings.verbosity=1; sc.settings.set_figure_params(dpi=70, facecolor='white')
    X=scipy.io.mmread('counts.mtx').T.tocsr()
    genes=pd.read_csv('genes.csv')['gene'].astype(str).values
    bc=pd.read_csv('barcodes.csv')['barcode'].astype(str).values
    meta=pd.read_csv('metadata.csv', index_col=0); umap=pd.read_csv('umap.csv', index_col=0)
    adata=ad.AnnData(X=X, obs=meta, var=pd.DataFrame(index=genes))
    adata.obs_names=bc
    adata.obsm['X_umap_authors']=umap.values   # UMAP original de los autores
    del X, meta, umap; gc.collect()
    # Detectar columnas de grupos
    def find_col(cs):
        for c in adata.obs.columns:
            if any(k in c.lower() for k in cs): return c
        return None
    col_cond =find_col(['disease','condition','group','sle','status','diagnosis'])
    col_time =find_col(['time','visit','day','week','treatment','point'])
    col_ctype=find_col(['celltype','cell_type','cell.type','annotation','ident'])
    print("condicion:",col_cond,"| timepoint:",col_time,"| tipo:",col_ctype)
    print(adata)


### 3B.6 · Guardar h5ad y liberar disco

In [ ]:
if MODO=="zenodo":
    import os, gc
    adata.write_h5ad(f"{PROJECT_DIR}/lupus_rituximab.h5ad")
    if os.path.exists('counts.mtx'): os.remove('counts.mtx')
    gc.collect(); print("Guardado.")


---
# 4 · Análisis y las 12 gráficas

Este bloque corre **en ambos modos** sobre `adata`. Genera las 12 salidas pedidas.


### 4.0 · Preparar marcadores según el modo
Un ayudante filtra los genes que sí existen en los datos (evita errores si falta alguno).


In [ ]:
import scanpy as sc, numpy as np, pandas as pd

def present(md_dict):
    out={k:[g for g in v if g in adata.var_names] for k,v in md_dict.items()}
    return {k:v for k,v in out.items() if v}

if MODO=="ejemplo":
    MARKERS = present({
        'CD14+ Mono':['FCN1','CD14'],'CD16+ Mono':['TCF7L2','FCGR3A','LYN'],
        'cDC2':['CST3','COTL1','LYZ','CLEC10A','FCER1A'],
        'Erythroblast':['MKI67','HBA1','HBB'],'Proerythroblast':['CDK6','SYNGR1','HBM','GYPA'],
        'NK':['GNLY','NKG7','CD247','TYROBP','KLRG1'],
        'Naive CD20+ B':['MS4A1','IL4R','IGHD','FCRL1','IGHM'],
        'Plasma cells':['MZB1','HSP90B1','PRDM1','IGKC','JCHAIN'],
        'CD4+ T':['CD4','IL7R','TRBC2'],'CD8+ T':['CD8A','CD8B','GZMK','CCL5','GZMB'],
        'T naive':['LEF1','CCR7','TCF7'],'pDC':['IL3RA','COBLL1','TCF4'],
    })
else:
    MARKERS = present({
        'T CD4':['CD3D','CD4','IL7R'],'T CD8':['CD3D','CD8A','GZMK'],
        'B':['MS4A1','CD79A','CD79B'],'NK':['GNLY','NKG7','KLRD1'],
        'Mono':['CD14','LYZ','FCGR3A'],'DC':['FCER1A','CST3'],
        'Plasma':['MZB1','JCHAIN','IGHG1'],
    })
print("Marcadores:", list(MARKERS.keys()))


### 4.1 · [Gráfica 1] Violines de QC
Número de genes por célula, conteos totales y % de conteos mitocondriales.


In [ ]:
adata.var['mt']  =adata.var_names.str.startswith('MT-')
adata.var['ribo']=adata.var_names.str.startswith(('RPS','RPL'))
adata.var['hb']  =adata.var_names.str.contains('^HB[^(P)]')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt','ribo','hb'], inplace=True, log1p=True)
sc.pl.violin(adata, ['n_genes_by_counts','total_counts','pct_counts_mt'],
             jitter=0.4, multi_panel=True)


### 4.2 · [Gráfica 2] Scatter de QC coloreado
Conteos totales vs. genes detectados, color = % mitocondrial.


In [ ]:
sc.pl.scatter(adata, 'total_counts', 'n_genes_by_counts', color='pct_counts_mt')

# Filtrado: en ejemplo (crudo) filtramos + dobletes; en zenodo ya viene curado
if MODO=="ejemplo":
    sc.pp.filter_cells(adata, min_genes=100)
    sc.pp.filter_genes(adata, min_cells=3)
    sc.pp.scrublet(adata, batch_key='sample', random_state=0)
    print("Dobletes:", int(adata.obs['predicted_doublet'].sum()))
else:
    sc.pp.filter_genes(adata, min_cells=3)   # limpieza suave de genes


### 4.3 · [Gráfica 3] Selección de características: normalizado vs no
Guardamos los conteos crudos, normalizamos + log1p, y marcamos los genes altamente variables (HVG).


In [ ]:
adata.layers['counts']=adata.X.copy()
sc.pp.normalize_total(adata); sc.pp.log1p(adata)
batch = 'sample' if 'sample' in adata.obs.columns else None
sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key=batch)
sc.pl.highly_variable_genes(adata)   # muestra normalizado vs no


### 4.4 · [Gráfica 4] PCA — PC1/PC2 y PC3/PC4
Coloreado por grupo y por % mitocondrial. Incluye la varianza explicada por componente.


In [ ]:
sc.tl.pca(adata, random_state=0)
color_group = 'sample' if 'sample' in adata.obs.columns else (col_cond or 'pct_counts_mt')
# color y dimensions se emparejan por posicion (zip): repetimos para PC1/2 y PC3/4
sc.pl.pca(adata,
          color=[color_group, color_group, 'pct_counts_mt', 'pct_counts_mt'],
          dimensions=[(0,1), (2,3), (0,1), (2,3)], ncols=2, size=3)
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)


### 4.5 · [Gráfica 5] Grafo de vecinos más cercanos
Construimos el grafo kNN y el UMAP; dibujamos las aristas del grafo sobre el UMAP.


In [ ]:
sc.pp.neighbors(adata, random_state=0)
sc.tl.umap(adata, random_state=0)
sc.pl.umap(adata, color=color_group, edges=True, edges_width=0.05,
           title='Grafo de vecinos mas cercanos')


### 4.6 · Clustering Leiden (base para el resto)
Tres resoluciones. El resto de gráficas usa `CLUSTER_KEY` (por defecto res 0.50).


In [ ]:
for res in [0.02, 0.5, 2.0]:
    sc.tl.leiden(adata, key_added=f'leiden_res_{res:4.2f}', resolution=res,
                 flavor='igraph', n_iterations=2, random_state=0)
    print(f"res {res}: {adata.obs[f'leiden_res_{res:4.2f}'].nunique()} clusters")


### 4.7 · [Gráfica 6] Filtrado visto con UMAP (métricas de QC)
Las métricas de calidad proyectadas sobre el UMAP, para ver si algún cluster es artefacto.


In [ ]:
qc_cols=['n_genes_by_counts','total_counts','pct_counts_mt']
if 'doublet_score' in adata.obs.columns: qc_cols.append('doublet_score')
sc.pl.umap(adata, color=qc_cols, ncols=2, size=3)


### 4.8 · [Gráfica 7] Anotación manual — números en cada grupo
Clusters con su número encima. En zenodo se muestra también la anotación de los autores.


In [ ]:
sc.pl.umap(adata, color=CLUSTER_KEY, legend_loc='on data', title='Clusters (numeros)')

if MODO=="ejemplo":
    cluster_to_celltype={'0':'Lymphocytes','1':'Monocytes','2':'Erythroid','3':'B Cells'}
    adata.obs['cell_type']=(adata.obs['leiden_res_0.02'].map(cluster_to_celltype)
                            .fillna('Unknown').astype('category'))
    sc.pl.umap(adata, color='cell_type', legend_loc='on data')
elif col_ctype:
    sc.pl.umap(adata, color=col_ctype, legend_loc='right margin',
               title='Anotacion de los autores')


### 4.9 · [Gráfica 8] Patrones de expresión de marcadores + dotplot
**Nota:** los patrones se generan solo con el **nombre del gen** (símbolo), no hace falta secuencia. scanpy busca el gen en la matriz y colorea según su expresión.


In [ ]:
# Dotplot de marcadores por cluster
sc.pl.dotplot(adata, MARKERS, groupby=CLUSTER_KEY, standard_scale='var')

# Patrones de expresion de algunos marcadores sobre el UMAP
genes_umap=[g for gs in MARKERS.values() for g in gs][:6]
sc.pl.umap(adata, color=genes_umap, ncols=3, size=3)


### 4.10 · Expresión diferencial (base para gráficas 9-11)
Test de Wilcoxon: genes que definen cada cluster.

**Importante:** excluimos los genes *housekeeping* (mitocondriales `MT-`, ribosomales `RPS`/`RPL` y hemoglobina `HB`) antes del test. Estos genes dominan el ranking como ruido técnico (aparecen "diferenciales" en muchos clusters) y no son marcadores biológicos útiles. El DE se calcula sobre `adata_de` (subconjunto sin esos genes); `adata` queda intacto.


In [ ]:
# Excluir housekeeping: dominan el DE como ruido y no son marcadores reales
hk = (adata.var_names.str.startswith(('MT-','RPS','RPL','MRPS','MRPL')) |
      adata.var_names.str.contains('^HB[^(P)]'))
adata_de = adata[:, ~hk].copy()
print(f"Genes housekeeping excluidos del DE: {int(hk.sum())} | genes usados: {adata_de.n_vars}")
sc.tl.rank_genes_groups(adata_de, groupby=CLUSTER_KEY, method='wilcoxon')
print("Expresion diferencial calculada sobre", CLUSTER_KEY)


### 4.11 · [Gráfica 9] Dotplot de genes diferencialmente expresados

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_de, groupby=CLUSTER_KEY,
                                standard_scale='var', n_genes=5)


### 4.12 · [Gráfica 10] Top-100 genes diferenciales — tabla + gráficas
**El punto más importante.** Exportamos un CSV con los 100 genes más diferenciales por cluster (ya sin housekeeping) y mostramos los rankings.


In [ ]:
de_all = sc.get.rank_genes_groups_df(adata_de, group=None)
top100 = de_all.groupby('group', group_keys=False).head(100)
out_csv = f"{PROJECT_DIR}/top100_DE_{MODO}.csv"
top100.to_csv(out_csv, index=False)
print(f"Guardado: {out_csv}  ({len(top100)} filas = 100 x {top100['group'].nunique()} clusters)")

# Vista rapida de los primeros por cluster
display(top100.groupby('group').head(5))

# Grafica de rankings (top genes por cluster)
sc.pl.rank_genes_groups(adata_de, n_genes=20, sharey=False)


### 4.13 · [Gráfica 11] Heatmap de expresión diferencial
Con pocos genes por cluster y ejes intercambiados (`swap_axes`) para que los nombres se lean.


In [ ]:
sc.pl.rank_genes_groups_heatmap(adata_de, n_genes=3, groupby=CLUSTER_KEY,
                                standard_scale='var', show_gene_labels=True,
                                swap_axes=True, figsize=(12,14))


### 4.14 · [Gráfica 12] Trayectoria (PAGA + pseudotiempo)
Reconstruye linajes: **PAGA** conecta los clusters según su similitud y el **pseudotiempo (DPT)** ordena las células a lo largo del linaje. Requiere una célula raíz (`ROOT_CLUSTER`); si es `None`, se elige automáticamente.


In [ ]:
# PAGA: grafo de conexiones entre clusters
sc.tl.paga(adata, groups=CLUSTER_KEY)
sc.pl.paga(adata, color=CLUSTER_KEY, title='PAGA: conexiones entre clusters')

# UMAP inicializado con PAGA (respeta la topologia de los linajes)
sc.tl.umap(adata, init_pos='paga', random_state=0)

# Pseudotiempo: elegir raiz
import numpy as np
root = ROOT_CLUSTER if ROOT_CLUSTER is not None else adata.obs[CLUSTER_KEY].value_counts().index[-1]
adata.uns['iroot'] = int(np.flatnonzero(adata.obs[CLUSTER_KEY]==str(root))[0])
sc.tl.dpt(adata)
print("Cluster raiz del pseudotiempo:", root)
sc.pl.umap(adata, color=[CLUSTER_KEY,'dpt_pseudotime'], legend_loc='on data', size=3)


---
# 5 · [MODO ZENODO] Figuras del artículo de lupus
> Solo corre si `MODO=="zenodo"`. Usa la anotación de los autores.


### 5.1 · Figura 1 — Subtipos de células B

In [ ]:
if MODO=="zenodo" and col_ctype:
    B_KW=['naive b','transitional','memory b','switched','abc','plasmablast','plasma','b cell','b_cell']
    isB=adata.obs[col_ctype].astype(str).str.lower().str.contains('|'.join(B_KW))
    adata_B=adata[isB].copy()
    print(f"Celulas B: {adata_B.n_obs:,}")
    print(adata_B.obs[col_ctype].value_counts())
    sc.pl.umap(adata_B, color=col_ctype, size=8, title='Subtipos de celulas B (Lupus)')


### 5.2 · Figura 2 — Volcano pre vs post-rituximab
Ajusta `PRE_LABEL`/`POST_LABEL` con los valores reales del timepoint (celda 3B.4/3B.5).


In [ ]:
if MODO=="zenodo":
    PRE_LABEL='Pretreatment'; POST_LABEL='Early post-treatment'   # <-- AJUSTAR
    CELLTYPE_SUBSET=None
    ad_de=adata
    if CELLTYPE_SUBSET and col_ctype:
        ad_de=adata[adata.obs[col_ctype].astype(str)==CELLTYPE_SUBSET]
    ad_de=ad_de[ad_de.obs[col_time].astype(str).isin([PRE_LABEL,POST_LABEL])].copy()
    print(ad_de.obs[col_time].value_counts())
    sc.tl.rank_genes_groups(ad_de, groupby=col_time, groups=[POST_LABEL],
                            reference=PRE_LABEL, method='wilcoxon')
    de=sc.get.rank_genes_groups_df(ad_de, group=POST_LABEL)


In [ ]:
if MODO=="zenodo":
    import matplotlib.pyplot as plt, numpy as np
    d=de.dropna(subset=['logfoldchanges','pvals_adj']).copy()
    d['nlp']=-np.log10(d['pvals_adj'].clip(lower=1e-300))
    up=(d['logfoldchanges']>1)&(d['pvals_adj']<0.05); dn=(d['logfoldchanges']<-1)&(d['pvals_adj']<0.05)
    plt.figure(figsize=(8,6))
    plt.scatter(d['logfoldchanges'],d['nlp'],s=6,c='lightgray')
    plt.scatter(d.loc[up,'logfoldchanges'],d.loc[up,'nlp'],s=8,c='#c0392b',label='Up')
    plt.scatter(d.loc[dn,'logfoldchanges'],d.loc[dn,'nlp'],s=8,c='#2e5f9a',label='Down')
    for _,r in d[up|dn].nlargest(15,'nlp').iterrows(): plt.text(r['logfoldchanges'],r['nlp'],r['names'],fontsize=7)
    plt.axvline(0,color='k',lw=.5); plt.axhline(-np.log10(0.05),color='k',ls='--',lw=.5)
    plt.xlabel(f'logFC ({POST_LABEL} - {PRE_LABEL})'); plt.ylabel('-log10 p'); plt.legend()
    plt.title('Volcano post vs pre-rituximab'); plt.tight_layout(); plt.show()


---
# 6 · Parte B — Redes regulatorias (SCENIC)

GRNBoost2 → cisTarget → AUCell. Corre en ambos modos; la matriz de entrada cambia (celda 6.2).


### 6.0 · Parche de compatibilidad de pySCENIC

pySCENIC 0.12.1 usa `np.object`, `np.bool`, etc., que numpy eliminó en versiones nuevas. Sin este parche, `pyscenic ctx` (cisTarget) falla con `AttributeError: module 'numpy' has no attribute 'object'` y devuelve **0 regulones**. Esta celda reemplaza esos alias obsoletos en los archivos de pySCENIC y ctxcore.


In [ ]:
import os, glob, re
import pyscenic, ctxcore
repl = [(r'\bnp\.object\b','object'), (r'\bnp\.bool\b','bool'),
        (r'\bnp\.int\b','int'), (r'\bnp\.float\b','float'), (r'\bnp\.str\b','str')]
parchados = 0
for pkg in (pyscenic, ctxcore):
    base = os.path.dirname(pkg.__file__)
    for f in glob.glob(os.path.join(base, '**', '*.py'), recursive=True):
        s = open(f, encoding='utf-8').read(); orig = s
        for pat, rep in repl:
            s = re.sub(pat, rep, s)
        if s != orig:
            open(f, 'w', encoding='utf-8').write(s); parchados += 1
print(f"Archivos parchados: {parchados}")

# Cinturon y tirantes: restaurar los alias en el numpy ya cargado en memoria
# (cubre el proceso actual; el parche en disco cubre el subprocess de 'pyscenic ctx')
import numpy as np
for _n, _o in [('object',object),('bool',bool),('int',int),('float',float),('str',str)]:
    if not hasattr(np, _n): setattr(np, _n, _o)
print("Parche np.object aplicado (necesario para que cisTarget funcione en Colab)")


### 6.1 · Descargar bases de datos de SCENIC

In [ ]:
import os, urllib.request
os.makedirs('scenic_data', exist_ok=True)
COM={
 'motifs.tbl':'https://resources.aertslab.org/cistarget/motif2tf/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl',
 'hg38_rankings.feather':'https://resources.aertslab.org/cistarget/databases/homo_sapiens/hg38/refseq_r80/mc_v10_clust/gene_based/hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather',
}
if MODO=="ejemplo":
    COM['expr_mat_tiny.loom']='https://raw.githubusercontent.com/aertslab/SCENICprotocol/master/example/expr_mat_tiny.loom'
    COM['test_TFs_tiny.txt'] ='https://raw.githubusercontent.com/aertslab/SCENICprotocol/master/example/test_TFs_tiny.txt'
else:
    COM['allTFs_hg38.txt']='https://resources.aertslab.org/cistarget/tf_lists/allTFs_hg38.txt'
for fn,url in COM.items():
    dest=f'scenic_data/{fn}'
    if not os.path.exists(dest):
        print("Descargando",fn); urllib.request.urlretrieve(url,dest)
    print(f"  {fn}: {os.path.getsize(dest)/1e6:.1f} MB")


### 6.2 · Preparar matriz + TFs según el modo

In [ ]:
import pandas as pd, numpy as np
if MODO=="ejemplo":
    import loompy
    with loompy.connect('scenic_data/expr_mat_tiny.loom') as ds:
        ex_matrix=pd.DataFrame(ds[:,:].T, index=ds.ca['CellID'], columns=ds.ra['Gene'])
    tf_names=pd.read_csv('scenic_data/test_TFs_tiny.txt', header=None).iloc[:,0].tolist()
    N_EST=500
else:
    import scanpy as sc
    if 'adata_B' not in dir():
        B_KW=['naive b','transitional','memory b','switched','abc','plasmablast','plasma','b cell','b_cell']
        isB=adata.obs[col_ctype].astype(str).str.lower().str.contains('|'.join(B_KW))
        adata_B=adata[isB].copy()
    ad_s=adata_B.copy()
    if SCENIC_DOWNSAMPLE and ad_s.n_obs>SCENIC_N_CELLS:
        sc.pp.subsample(ad_s, n_obs=SCENIC_N_CELLS, random_state=0)
    print("Celulas B para SCENIC:", ad_s.n_obs)
    sc.pp.highly_variable_genes(ad_s, n_top_genes=min(SCENIC_N_GENES, ad_s.n_vars-1))
    ad_s=ad_s[:, ad_s.var.highly_variable].copy()
    Xd=ad_s.X.toarray() if hasattr(ad_s.X,'toarray') else np.asarray(ad_s.X)
    ex_matrix=pd.DataFrame(Xd, index=ad_s.obs_names.astype(str), columns=ad_s.var_names.astype(str))
    all_tfs=pd.read_csv('scenic_data/allTFs_hg38.txt', header=None).iloc[:,0].tolist()
    tf_names=[t for t in all_tfs if t in ex_matrix.columns]; N_EST=200
    if not SCENIC_DOWNSAMPLE: print("AVISO: sin downsample -> puede tardar horas.")
print(f"Matriz: {ex_matrix.shape} | TFs: {len(tf_names)}")


### 6.3 · GRNBoost2 (scikit-learn)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
tfs_in=[t for t in tf_names if t in ex_matrix.columns]
X_tfs=ex_matrix[tfs_in].values
targets=[g for g in ex_matrix.columns if g not in tfs_in]
records=[]
for i,tg in enumerate(targets):
    y=ex_matrix[tg].values
    if y.std()==0: continue
    gbm=GradientBoostingRegressor(n_estimators=N_EST, max_depth=3, random_state=42)
    gbm.fit(X_tfs, y)
    for tf,imp in zip(tfs_in, gbm.feature_importances_):
        if imp>0: records.append({'TF':tf,'target':tg,'importance':imp})
    if (i+1)%200==0: print(f"  {i+1}/{len(targets)}")
adjacencies=pd.DataFrame(records).sort_values('importance',ascending=False)
adjacencies.to_csv('scenic_data/adjacencies.tsv', sep='\t', index=False)
print("Relaciones TF-gen:", len(adjacencies))


### 6.4 · cisTarget

In [ ]:
import loompy, numpy as np
LOOM='scenic_data/expr_mat_tiny.loom' if MODO=="ejemplo" else 'scenic_data/expr_bcells.loom'
if MODO=="zenodo":
    loompy.create(LOOM, ex_matrix.T.values, {'Gene':np.array(ex_matrix.columns)}, {'CellID':np.array(ex_matrix.index)})
!pyscenic ctx scenic_data/adjacencies.tsv scenic_data/hg38_rankings.feather --annotations_fname scenic_data/motifs.tbl --expression_mtx_fname {LOOM} --output scenic_data/regulons.csv --num_workers 2
try:
    df=pd.read_csv('scenic_data/regulons.csv', index_col=[0,1], header=[0,1]); n_regulons=len(df)
except Exception:
    n_regulons=0
print("Regulones:", n_regulons)


### 6.5 · AUCell (con fallback)

In [ ]:
from pyscenic.aucell import aucell as pyscenic_aucell
from ctxcore.genesig import GeneSignature
import ast
signatures=[]
if n_regulons>0:
    dfc=pd.read_csv('scenic_data/regulons.csv', header=[0,1], index_col=[0,1])
    dfc.columns=[' '.join(c).strip() for c in dfc.columns]; dfc=dfc.reset_index()
    tgt=next((c for c in dfc.columns if 'TargetGenes' in c),None)
    tfc=next((c for c in dfc.columns if c in ('TF','level_0')),None)
    for _,row in dfc.iterrows():
        try: gs=[t[0] for t in ast.literal_eval(str(row[tgt]))]
        except Exception: gs=[]
        if gs: signatures.append(GeneSignature(name=f"{row[tfc]}(+)", gene2weight={g:1.0 for g in gs}))
else:
    print("Fallback: regulones desde el GRN")
    for tf,grp in adjacencies.groupby('TF'):
        signatures.append(GeneSignature(name=f"{tf}(+)", gene2weight=dict(zip(grp['target'],grp['importance']))))
print("Firmas:", len(signatures))
auc_matrix=pyscenic_aucell(ex_matrix, signatures, num_workers=1)
print("AUCell:", auc_matrix.shape)


### 6.6 · Visualizar regulones

In [ ]:
import scanpy as sc
asc=sc.AnnData(X=auc_matrix.values, obs=pd.DataFrame(index=auc_matrix.index.astype(str)),
               var=pd.DataFrame(index=auc_matrix.columns.astype(str)))
sc.pp.neighbors(asc, random_state=42); sc.tl.umap(asc, random_state=42)
sc.tl.leiden(asc, flavor='igraph', n_iterations=2, random_state=42)
sc.pl.umap(asc, color='leiden', title=f'Regulones activos ({MODO})')
auc_matrix.to_csv(f"{PROJECT_DIR}/scenic_auc_{MODO}.csv")
print("Guardado.")


---
# 7 · Resumen

Se generaron las **12 gráficas** (sección 4) para el dataset elegido, más las figuras del artículo (modo zenodo) y SCENIC.

**Salidas guardadas:** `adata` procesado, `top100_DE_<modo>.csv`, `scenic_auc_<modo>.csv`.

**Parámetros** (celda 1): `MODO`, `N_CELLS_MAX`, `SCENIC_DOWNSAMPLE`, `ROOT_CLUSTER`, `CLUSTER_KEY`.

**Siguiente fase:** CloudPred — predicción clínica SLE vs sano por paciente.

---
Notebook versión **1.1.1**
